<a id="top"></a>
# House Sales: preparazione dati e primo modello di regressione (esercitazione)

Nel modulo 1 hai usato il dataset House Sales per inquadrare il problema: stimare il **prezzo di vendita** delle case in una certa area.  
Nel modulo 2, con il caso Bank Marketing, hai lavorato sulla preparazione di un dataset per un problema di **classificazione binaria**.

Qui resti nel contesto naturale di House Sales: **regressione sul prezzo**.  
L’obiettivo è replicare in versione compatta la pipeline di preparazione dati vista nel marketing bancario e collegarla a un primo modello lineare.

<a id="obiettivo-esercitazione"></a>
## Obiettivo dell'esercitazione

Nel caso Bank Marketing ti sei concentrato su una **classificazione**: prevedere se un cliente sottoscrive o meno un deposito (output binario 0/1).  
Nel caso House Sales torni al problema di **regressione**: il modello prevede un **numero continuo** (il prezzo), non una classe.

Riassumendo in due righe la differenza:

- **Classificazione**: l’output è una categoria (es. _sì/no_, _classe A/B/C_).  
- **Regressione**: l’output è un valore continuo (es. prezzo in euro).

In questa esercitazione:

- preparerai le feature in modo coerente con quanto fatto sul marketing (imputazione + normalizzazione robusta);
- allenerai un modello lineare (`LinearRegression`) per prevedere il prezzo;



<a id="caricamento-dataset"></a>
## Caricamento del dataset House Sales

Userai lo stesso dataset del notebook House Sales del modulo 1.  
Il dataset è su OpenML (id 42092) e si può scaricare con `fetch_openml` di scikit-learn.

- `fetch_openml` scarica un dataset da OpenML e restituisce un oggetto da cui puoi ricavare un DataFrame.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html

Nel blocco seguente carichi i dati in un DataFrame `house` e dai un’occhiata alle prime righe.


In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  # House sales
house = raw.frame.copy()

print("Dimensioni del dataset (righe, colonne):", house.shape)
house.head()


Dimensioni del dataset (righe, colonne): (21613, 20)


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


# Step 1: Preparazione del dato

Nel primo step dovrai preparare il dato:
1. separare X e y 
2. rimuovere le features inutilizzabili (date, zipcode, lat, long, yr_built, yr_renovated)
3. train test split (30% di test)


<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

In questo contesto ci accontentiamo di usare le variabili così come sono per faciilitare le prossime fasi. Ma la pratica del *feature engineering* prevede l'elaborazione di nuove variabili più utiili:
- yr_built -> età 
- yr_nenovated -> renovated?
- sqft_above, sqft_basement-> metri quadri totali  

</div>

In [ ]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
set_config(transform_output="pandas")
import pandas as pd
from sklearn.datasets import fetch_openml

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  # House sales
house = raw.frame.copy()

print("Dimensioni del dataset (righe, colonne):", house.shape)
house.head()

#1. separare X e y 
# Dividi il dataset in set di addestramento e test
X = house.drop(columns="price")
y = house["price"]

#2. rimuovere le features inutilizzabili (date, zipcode, lat, long, yr_built, yr_renovated)
X = X.drop(columns=["date", "zipcode", "lat", "long", "yr_built", "yr_renovated"])


#3. suddividere in train e test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# NB: test_size=0.3! 

Dimensioni del dataset (righe, colonne): (21613, 20)


In [34]:
#3. suddividere in train e test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# NB: test_size=0.3!

## Step 2 : Pipeline di preprocessing 

Dovrai creare una pipeline di scikit composta da `SimpleImputer` e `RobustScaler`. In questo caso **non** ti servirà usare `ColumnTrasformer` perché tutte le features sono numeriche (verifica). 

Usa la mediana per l'imputazione di tutte le features:
- ("imputer", SimpleImputer(strategy="median"))
- https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html

In [11]:
# verifica che siano davvero tutte numeriche
print(X_train.dtypes)

bedrooms           int64
bathrooms        float64
sqft_living        int64
sqft_lot           int64
floors           float64
waterfront         int64
view               int64
condition          int64
grade              int64
sqft_above         int64
sqft_basement      int64
sqft_living15      int64
sqft_lot15         int64
dtype: object


In [14]:
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn import set_config
from sklearn.model_selection import train_test_split
set_config(transform_output="pandas")
import pandas as pd
from sklearn.datasets import fetch_openml

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  # House sales
house = raw.frame.copy()

print("Dimensioni del dataset (righe, colonne):", house.shape)
house.head()

#1. separare X e y 
# Dividi il dataset in set di addestramento e test
X = house.drop(columns="price")
y = house["price"]

#2. rimuovere le features inutilizzabili (date, zipcode, lat, long, yr_built, yr_renovated)
X = X.drop(columns=["date", "zipcode", "lat", "long", "yr_built", "yr_renovated"])


#3. suddividere in train e test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# NB: test_size=0.3! 
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

In [32]:
#inserisci alcuni dati mancanti randomcon iloc (scusate ho dimentica di inserirli)
X_train.iloc[0:-1000, 0] = None
# Non ci sono più valori mancanti con l'imputer!
print("Valori mancanti in X_train:", X_train.isnull().sum().sum())
X_train_processed = preprocess.fit_transform(X_train)
print("Valori mancanti in X_train_processed:", X_train_processed.isnull().sum().sum())

Valori mancanti in X_train: 15128
Valori mancanti in X_train_processed: 0


## Step 3: Aggiungi anche `LinearRegression` come estimator in fondo alla pipeline. Produci delle previsioni sul test

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn import set_config
from sklearn.model_selection import train_test_split
set_config(transform_output="pandas")
import pandas as pd
from sklearn.datasets import fetch_openml

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  # House sales
house = raw.frame.copy()

print("Dimensioni del dataset (righe, colonne):", house.shape)
house.head()

#1. separare X e y 
# Dividi il dataset in set di addestramento e test
X = house.drop(columns="price")
y = house["price"]

#2. rimuovere le features inutilizzabili (date, zipcode, lat, long, yr_built, yr_renovated)
X = X.drop(columns=["date", "zipcode", "lat", "long", "yr_built", "yr_renovated"])


#3. suddividere in train e test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# NB: test_size=0.3! 
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

model_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression(iterations=1000, random_state=42))
])
# facciamo il training del modello
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['bedrooms','bathrooms','sqft_living',...,'sqft_basement','sqft_living15', 'sqft_lot15']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitt

In [ ]:
# prevediamo sui dati di test
model_pipeline.predict(X_test)

array([ 553966.33266221,  752322.13729963, 1251867.86602962, ...,
        281733.06135238,  669582.5201628 ,  608790.56162953],
      shape=(6484,))

## Extra: se hai concluso puoi provare ad esplorare:

1. cambia strategia di imputing (mean invece di median)
2. cambia modello, from sklearn.ensemble import GradientBoostingRegressor (info: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html#sklearn.ensemble.GradientBoostingRegressor)

Molto avanzato: 
1. Fai uno stacking dei due modelli che hai allenato (https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html)
